# Batch Inference Runner for 2D Cell Division Detection

This notebook is the hands-on companion to our project. Run it when you want a guided, reproducible way to execute the full DARE2d pipeline (inference -> consensus -> plots) on your own `.tif` stacks.

**Workflow**
1. Update the path variables in Cell 2 (or export `DARE2D_BASE_DIR`) and activate the `dare2d` environment.
2. Run the remaining cells in order: data discovery, multi-model inference, postprocessing, and plotting.
3. Inspect the generated folders (`output/`, `postprocessed_results/`, `analysis_plots/`).

> Need installation details, model descriptions, or troubleshooting tips? See `README.md` so this notebook can stay lean.

In [6]:
# Setup: Import Libraries and Configure Paths

# Import required Python libraries
import os
import subprocess
import glob
from pathlib import Path

# Configuration Section
# Modify these paths according to your project setup

# base_dir: Root directory of the DARE2d project
default_base_dir = Path.cwd().resolve()
base_dir = Path(os.environ.get("DARE2D_BASE_DIR", str(default_base_dir)))

# Define subdirectories relative to base_dir
input_dir = base_dir / "input"  # Directory containing input .tif images
output_dir = base_dir / "output"  # Directory for saving inference results
reg_dir = base_dir / "regression_checkpoints"  # Regression model checkpoints
seg_dir = base_dir / "segmentation_checkpoints"  # Segmentation model checkpoints

# Verify that the base directory exists
if not base_dir.exists():
    raise FileNotFoundError(f"Base directory not found: {base_dir}. Please update base_dir to the correct path.")

print(f"Project root: {base_dir}")
print(f"Input directory: {input_dir}")
print(f"Output directory: {output_dir}")
print("Setup completed successfully.")

Project root: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d
Input directory: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\input
Output directory: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\output
Setup completed successfully.


## Data Discovery

In this section, we identify all input images that will be processed. The script expects images in .tif format placed in the input directory.

The following code:
- Scans the input directory for .tif files
- Displays the number of images found
- Lists the image names for verification

In [7]:
# Data Discovery: Find Input Images

# Discover all .tif images in the input directory
input_images = glob.glob(os.path.join(input_dir, "*.tif"))

if not input_images:
    print("Error: No .tif images found in the input directory!")
    print(f"Input directory: {input_dir}")
    print("Please ensure input images are placed in the correct directory.")
    # In a notebook, you might want to raise an error or stop execution
    raise FileNotFoundError("No input images found")

print(f"Found {len(input_images)} input images:")
for img_path in input_images:
    image_name = Path(img_path).stem
    print(f"  - {image_name}")

# Store image names for later use
image_names = [Path(img).stem for img in input_images]

Found 1 input images:
  - unwoundh18h22_detected.tif (green)


## Batch Inference Execution

Every image discovered in the previous cell is paired with each of the eight model sets. The loop simply verifies the checkpoints, creates an `output_dir/{image_name}_{n}` folder, and calls `scripts.inference.multistage_detection2d`.

> For background on the model architecture, training splits, or checkpoint organization, keep the README handy—this notebook just wires everything together.

In [8]:
# Batch Inference: Process All Images with Multiple Model Sets

# Number of model sets to use (1-8)
num_model_sets = 8

# Process each image
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Processing image: {image_name}")
    print(f"{'='*60}")

    # Run inference with each model set
    for n in range(1, num_model_sets + 1):
        print(f"\n--- Model Set {n} ---")

        # Construct paths to model checkpoints
        # Assumes organization: checkpoints_set_{n}_all_but_target/best.h5
        reg_checkpoint = os.path.join(reg_dir, f"checkpoints_set_{n}_all_but_target", "best.h5")
        seg_checkpoint = os.path.join(seg_dir, f"checkpoints_set_{n}_all_but_target", "best.h5")

        # Verify model files exist
        if not os.path.exists(reg_checkpoint):
            print(f"Warning: Regression checkpoint not found: {reg_checkpoint}")
            continue
        if not os.path.exists(seg_checkpoint):
            print(f"Warning: Segmentation checkpoint not found: {seg_checkpoint}")
            continue

        # Create output directory for this image-model combination
        output_folder = os.path.join(output_dir, f"{image_name}_{n}")
        os.makedirs(output_folder, exist_ok=True)

        # Construct the inference command
        cmd = [
            "python",
            "-m", "scripts.inference.multistage_detection2d",
            "--regression", reg_checkpoint,
            "--segmentation", seg_checkpoint,
            "--img", img_path,
            "--output", output_folder
        ]

        print(f"Running command: {' '.join(cmd)}")

        try:
            # Execute the inference
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"✓ Successfully completed inference for {image_name} with model set {n}")
            # Optionally print stdout if needed: print(result.stdout)

        except subprocess.CalledProcessError as e:
            print(f"✗ Error running inference for {image_name} with model set {n}")
            print(f"Error details: {e}")
            print(f"Stderr: {e.stderr}")
            continue

print(f"\n{'='*60}")
print("Batch inference completed!")
print(f"Results saved in: {output_dir}")
print("Each image has subdirectories for each model set (1-8)")
print(f"Total combinations processed: {len(input_images)} images × {num_model_sets} model sets = {len(input_images) * num_model_sets}")
print(f"{'='*60}")


Processing image: unwoundh18h22_detected.tif (green)

--- Model Set 1 ---
Running command: python -m scripts.inference.multistage_detection2d --regression C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\regression_checkpoints\checkpoints_set_1_all_but_target\best.h5 --segmentation C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\segmentation_checkpoints\checkpoints_set_1_all_but_target\best.h5 --img C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\input\unwoundh18h22_detected.tif (green).tif --output C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\output\unwoundh18h22_detected.tif (green)_1
✓ Successfully completed inference for unwoundh18h22_detected.tif (green) with model set 1

--- Model Set 2 ---
Running command: python -m scripts.inference.multistage_detection2d --regression C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\regression_checkpoints\checkpoints_set_2_all_but_target\best.h5 --segmentation C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\segmentation_chec

## Postprocessing and Consensus Generation

Once inference completes, we hand the per-model outputs to `scripts/postprocessing/main.py` to cluster detections, perform temporal deduplication, and write timestamped consensus folders inside `postprocess_dir`.

> Detailed parameter explanations (e.g., `eps`, `min_models`, `angle_mode`) and standalone CLI usage examples are documented in the README; the notebook focuses on running the batch pipeline end-to-end.

In [9]:
# Postprocessing: Aggregate Multi-Model Results

# Create directory for postprocessing outputs
postprocess_dir = os.path.join(base_dir, "postprocessed_results")
os.makedirs(postprocess_dir, exist_ok=True)

# Track resolved output directories (handles timestamped folders created by script)
postprocess_output_dirs = {}

# Postprocessing parameters
eps = 10  # Clustering epsilon (spatial distance threshold)
min_models = 1  # Minimum number of models required for consensus
num_models = 8  # Total number of models
angle_mode = "auto"  # Angle selection mode

print(f"Postprocessing outputs will be saved to: {postprocess_dir}")

# Process each image with postprocessing
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Postprocessing image: {image_name}")
    print(f"{'='*60}")

    # Define base directory for this image's postprocessed results (timestamp appended by script)
    image_postprocess_dir = os.path.join(postprocess_dir, image_name)

    # Snapshot existing directories to detect newly created timestamped folders
    base_pattern = f"{image_name}*"
    existing_dirs = {d.name for d in Path(postprocess_dir).glob(base_pattern) if d.is_dir()}

    # Construct postprocessing command
    cmd = [
        "python",
        "scripts/postprocessing/main.py",
        "--output_root", str(output_dir),
        "--image_name", image_name,
        "--image_stack", str(img_path),
        "--save_dir", image_postprocess_dir,
        "--eps", str(eps),
        "--min_models", str(min_models),
        "--num_models", str(num_models),
        "--angle_mode", angle_mode
    ]

    print(f"Running postprocessing command: {' '.join(cmd)}")

    try:
        # Execute postprocessing
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✓ Successfully completed postprocessing for {image_name}")
        
        # Resolve actual output folder (script appends timestamp to avoid overwriting)
        candidates = [d for d in Path(postprocess_dir).glob(base_pattern) if d.is_dir()]
        new_dirs = [d for d in candidates if d.name not in existing_dirs]
        if new_dirs:
            final_dir = max(new_dirs, key=lambda p: p.stat().st_mtime)
        else:
            final_dir = max(candidates, key=lambda p: p.stat().st_mtime, default=Path(image_postprocess_dir))
        postprocess_output_dirs[image_name] = str(final_dir)
        print(f"Postprocessing outputs stored in: {final_dir}")

    except subprocess.CalledProcessError as e:
        print(f"✗ Error in postprocessing for {image_name}")
        print(f"Error details: {e}")
        print(f"Stderr: {e.stderr}")
        continue

print(f"\n{'='*60}")
print("Postprocessing completed for all images!")
print(f"Results saved in: {postprocess_dir}")
print("Each image has its own subfolder with consensus results.")

Postprocessing outputs will be saved to: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\postprocessed_results

Postprocessing image: unwoundh18h22_detected.tif (green)
Running postprocessing command: python scripts/postprocessing/main.py --output_root C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\output --image_name unwoundh18h22_detected.tif (green) --image_stack C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\input\unwoundh18h22_detected.tif (green).tif --save_dir C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\postprocessed_results\unwoundh18h22_detected.tif (green) --eps 10 --min_models 1 --num_models 8 --angle_mode auto
✓ Successfully completed postprocessing for unwoundh18h22_detected.tif (green)
Postprocessing outputs stored in: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\postprocessed_results\unwoundh18h22_detected.tif (green)_20260123_120729

Postprocessing completed for all images!
Results saved in: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\post

## Quality Analysis and Visualization

This step reuses the resolved postprocessing folders to call `scripts/postprocessing/plots.py`, generating per-image figures and CSVs inside `analysis_plots/{image_name}`.

> For descriptions of each plot/metric, refer to the README. The notebook keeps things minimal by just orchestrating the command.

In [10]:
# Quality Analysis and Visualization

# Create directory for plots and analysis
plots_dir = os.path.join(base_dir, "analysis_plots")
os.makedirs(plots_dir, exist_ok=True)

print(f"Analysis plots will be saved to: {plots_dir}")

# Lookup table populated during postprocessing (handles timestamped folders)
postprocess_dirs = globals().get("postprocess_output_dirs", {})

# Generate plots for each postprocessed image
for image_name in image_names:
    image_postprocess_dir = Path(postprocess_dirs.get(image_name, os.path.join(postprocess_dir, image_name)))

    # Check if postprocessing was successful for this image
    summary_csv = image_postprocess_dir / f"post_processed_{image_name}_summary.csv"
    processed_tiff = image_postprocess_dir / f"post_processed_{image_name}.tiff"

    if not summary_csv.exists():
        print(f"Warning: Summary CSV not found for {image_name}, skipping plots")
        continue

    if not processed_tiff.exists():
        print(f"Warning: Processed TIFF not found for {image_name}, skipping plots")
        continue

    print(f"\nGenerating plots for {image_name}...")

    # Create plots subdirectory for this image
    image_plots_dir = Path(plots_dir) / image_name
    image_plots_dir.mkdir(parents=True, exist_ok=True)

    # Construct plots command
    cmd = [
        "python",
        "scripts/postprocessing/plots.py",
        "--summary", str(summary_csv),
        "--tiff", str(processed_tiff),
        "--plots_dir", str(image_plots_dir)
    ]

    print(f"Running plots command: {' '.join(cmd)}")

    try:
        # Execute plots generation
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✓ Successfully generated plots for {image_name}")

    except subprocess.CalledProcessError as e:
        print(f"✗ Error generating plots for {image_name}")
        print(f"Error details: {e}")
        print(f"Stderr: {e.stderr}")
        continue

print(f"\n{'='*60}")
print("Quality analysis and visualization completed!")
print(f"Plots and metrics saved in: {plots_dir}")
print("Each image has its own subfolder with analysis results.")
print(f"{'='*60}")

Analysis plots will be saved to: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\analysis_plots

Generating plots for unwoundh18h22_detected.tif (green)...
Running plots command: python scripts/postprocessing/plots.py --summary C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\postprocessed_results\unwoundh18h22_detected.tif (green)_20260123_120729\post_processed_unwoundh18h22_detected.tif (green)_summary.csv --tiff C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\postprocessed_results\unwoundh18h22_detected.tif (green)_20260123_120729\post_processed_unwoundh18h22_detected.tif (green).tiff --plots_dir C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\analysis_plots\unwoundh18h22_detected.tif (green)
✓ Successfully generated plots for unwoundh18h22_detected.tif (green)

Quality analysis and visualization completed!
Plots and metrics saved in: C:\Users\Qazi\Documents\PhD_Y1\Divisions_2D\DARE2d\analysis_plots
Each image has its own subfolder with analysis results.
